In [60]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.base import clone
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)


<b>Production [tons], Yield [kg/ha], Area [ha]</b>

In [20]:
data = pd.read_csv("Production_Crops_Livestock_E_All_Data.csv")
data2 = pd.read_csv("FAO_PIN_Merged.csv")

/tmp/ipykernel_889/344734770.py:1: DtypeWarning: Columns (11,14,17,20,23,26,29,32,35,38,41,44,47,50,53,56,59,62,65,68,71,74,77,80,83,86,89,92,95,98,101,104,107,110,113,116,119,122,125,128,131,134,137,140,143,146,149,152,155,158,161,164,167,170,173,176,179,182,185,188,191,194,197,200) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("Production_Crops_Livestock_E_All_Data.csv")


In [21]:
Areas = [
    'Africa',
    'Americas',
    'Asia',
    'Australia and New Zealand',
    'Central America',
    'Central Asia',
    'China, mainland',
    'Other non-specified areas',
    'Eastern Africa',
    'Eastern Asia',
    'Eastern Europe',
    'Europe',
    'European Union (27)',
    'Land Locked Developing Countries',
    'Least Developed Countries',
    'Low Income Food Deficit Countries',
    'Micronesia (Federated States of)',
    'Middle Africa',
    'Net Food Importing Developing Countries',
    'Northern Africa',
    'Northern America',
    'Northern Europe',
    'Oceania',
    'Polynesia',
    'Small Island Developing States',
    'South America',
    'South-eastern Asia',
    'Southern Africa',
    'Southern Asia',
    'Southern Europe',
    'Western Africa',
    'Western Asia',
    'Western Europe',
    'World'
]

In [22]:
#PREFILTERING
data2 = data2[~data2["Country or Area"].isin(Areas)] # leaving only countries from dataset 2
Countries = list(data2["Country or Area"].unique()) # countries and columns filtered from the main data set
data = data[data["Area"].isin(Countries)] # leave just countries in main data

data = data.drop(columns=['Area Code (M49)', 'Area Code', 'Item Code (CPC)', 'Item Code', 'Element Code', 'Unit']) #unnecesary columns

years = list(range(1961, 2025))

for year in years:
    str_year = str(year)
    data = data.drop(columns="Y"+str_year+"F")
    data = data.drop(columns="Y"+str_year+"N")

year_cols = [col for col in data.columns if col.startswith("Y")]

data = data.melt( 
    id_vars=["Area", "Item", "Element"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Value"
) # datased is converted from years in columns to year in rows and all values are in one column

data["Year"] = (
    data["Year"]
    .str.replace("Y", "")
    .astype(int)
)

wanted_elements = [
    "Production",
    "Area harvested",
    "Yield"
]

data = data[
    data["Element"].isin(wanted_elements)
]

# CONVERT LONG → WIDE
data = (
    data.pivot_table(
        index=[
            "Area",
            "Year",
            "Item"
        ],
        columns="Element",
        values="Value",
        aggfunc="mean"
    )
    .reset_index()
)

# REMOVE EXTRA COLUMN NAME
data.columns.name = None

# data.dropna(inplace=True)

print(data.head())

# data.to_csv("prefilt_data.csv", index=False)

          Area  Year                            Item  Area harvested  \
0  Afghanistan  1961               Almonds, in shell             0.0   
1  Afghanistan  1961                          Apples          2220.0   
2  Afghanistan  1961                        Apricots          4820.0   
3  Afghanistan  1961                          Barley        350000.0   
4  Afghanistan  1961  Beef and Buffalo Meat, primary             NaN   

   Production   Yield  
0         0.0     NaN  
1     15100.0  6801.8  
2     32000.0  6639.0  
3    378000.0  1080.0  
4     43000.0     NaN  


In [23]:
corr_matrix = data.corr(numeric_only=True)
corr_matrix["Production"].sort_values(ascending=False) # Area harvested is most correlated with production and yield which is expected
# YIELD = PRODUCTION/AREA But yield is directly calculated from area and production 
# highly intercorrelated
# production should be removed ?? 

# since yield is directly calculated from area and production, it should be removed to avoid data leakage

# data = data.drop(columns=['Production']) --- IGNORE ---

Production        1.000000
Area harvested    0.591832
Year              0.027695
Yield             0.015570
Name: Production, dtype: float64

In [24]:
# "df" are temporary 
# "data" are derived from main data line

df_rank = data.groupby(
    ["Area", "Item"]
)["Production"].mean().reset_index()

df_rank["rank"] = df_rank.groupby(
    "Area"
)["Production"].rank(
    ascending=False,
    method="first"
)

# select top 5
df_top5 = df_rank[df_rank["rank"] <= 5]

data_top5 = data.merge(
    df_top5[["Area", "Item"]],
    on=["Area", "Item"],
    how="inner"
)


In [25]:
data_co = data[data["Item"].isin(["Coffee, green", "Cocoa beans"])]

In [26]:
# DATASETS FOR ANALYSIS - everything, coffe and cocoa, top 5 items in each country
# print(data)
# print(data_co)
# print(data_top5)

# data.to_csv("data.csv", index=False)
# data.to_csv("data_co.csv", index=False)
# data.to_csv("data_top5.csv", index=False)

In [27]:
# data_co = data_co.drop(columns='Production')
data_cof = data_co[data_co["Item"] == "Coffee, green"]
data_coc = data_co[data_co["Item"] == "Cocoa beans"]
data_cof = data_cof.drop(columns='Item')
data_coc = data_coc.drop(columns='Item')
# data_cof = data_cof.drop(columns='Production')
# data_coc = data_coc.drop(columns='Production')

COFFEE

In [28]:
# chosen_prod = 

df = data_cof.copy() #copy data to df for processing
df = data_cof.drop(columns='Production') # drop production
df = df[df["Year"] <= 2024]

df = df.sort_values(
    ["Area", "Year"]
)

def add_lags(df, cols, lags=[1,2,3]): #function for engineering lag features

    df = df.copy()

    for col in cols:

        for lag in lags:

            df[f"{col}_lag{lag}"] = (
                df
                .groupby(["Area"])[col]
                .shift(lag)
            )

    return df 


def add_rolling(df, col): # function for engineering rolling mean features

    df = df.copy()

    # for col in cols: # GPT not so smart ha

    df[f"{col}_roll3"] = (
        df
        .groupby(["Area"])[col]
        .transform(
            lambda x:
            x.shift(1)
                .rolling(3,min_periods=1)
                .mean()
        )
    )

    df[f"{col}_roll5"] = (
        df
        .groupby(["Area"])[col]
        .transform(
            lambda x:
            x.shift(1)
                .rolling(5,min_periods=1)
                .mean()
        )
    )

    return df


def add_growth(df, cols): # function for engineering growth rate features

    df = df.copy()

    for col in cols:

        df[f"{col}_growth"] = (
            df
            .groupby(["Area"])[col]
            .pct_change()
        )

    return df

# apply feature funcs
base_cols = [ 
"Yield",
"Area harvested",
]

# df = df.dropna(subset=base_cols) # drop rows with missing values in base columns before applying feature engineering functions

df = add_lags(df, base_cols)

# df = df.dropna()

df = add_rolling(df, base_cols[0])
df = add_rolling(df, base_cols[1])

# df = df.dropna()

df = add_growth(df, base_cols)

# df = df.dropna()

/tmp/ipykernel_889/1618582373.py:68: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()
/tmp/ipykernel_889/1618582373.py:68: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


In [29]:
categorical_features = [
    "Area",
]

numeric_features = [

    "Year",

    "Yield_lag1",
    "Yield_lag2",
    "Yield_lag3",

    "Area harvested_lag1",
    "Area harvested_lag2",
    "Area harvested_lag3",

    "Yield_roll3",
    "Yield_roll5",

    "Area harvested_roll3",
    "Area harvested_roll5",

    "Yield_growth",
    "Area harvested_growth"
]

In [30]:
df

,Area,Year,Area harvested,Yield,Yield_lag1,Yield_lag2,Yield_lag3,Area harvested_lag1,Area harvested_lag2,Area harvested_lag3,Yield_roll3,Yield_roll5,Area harvested_roll3,Area harvested_roll5,Yield_growth,Area harvested_growth
20451,Angola,1961,350000.0,481.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20534,Angola,1962,500000.0,370.0,481.7,NaN,NaN,350000.0,NaN,NaN,481.700000,481.700,350000.000000,350000.0,-0.231887,0.428571
20617,Angola,1963,500000.0,336.6,370.0,481.7,NaN,500000.0,350000.0,NaN,425.850000,425.850,425000.000000,425000.0,-0.090270,0.000000
20700,Angola,1964,500000.0,396.4,336.6,370.0,481.7,500000.0,500000.0,350000.0,396.100000,396.100,450000.000000,450000.0,0.177659,0.000000
20783,Angola,1965,500000.0,410.0,396.4,336.6,370.0,500000.0,500000.0,500000.0,367.666667,396.175,500000.000000,462500.0,0.034309,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1122597,Zimbabwe,2020,573.0,1010.5,979.7,1086.0,1095.5,542.0,453.0,498.0,1053.733333,1076.180,497.666667,515.0,0.031438,0.057196
1122742,Zimbabwe,2021,676.0,899.4,1010.5,979.7,1086.0,573.0,542.0,453.0,1025.400000,1055.360,522.666667,539.8,-0.109946,0.179756
1122887,Zimbabwe,2022,681.0,1000.0,899.4,1010.5,979.7,676.0,573.0,542.0,963.200000,1014.220,597.000000,548.4,0.111852,0.007396
1123032,Zimbabwe,2023,685.0,971.5,1000.0,899.4,1010.5,681.0,676.0,573.0,969.966667,995.120,643.333333,585.0,-0.028500,0.005874


In [31]:
#pipeline ahitecture

preprocessor = ColumnTransformer(

    transformers=[

        (
            "cat",

            Pipeline([

                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="missing"
                    )
                ),

                (
                    "ohe",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )

            ]),

            categorical_features
        ),

        (
            "num",

            Pipeline([

                (
                    "imputer",
                    SimpleImputer(
                        strategy="median",
                        # fill_value=0
                    )
                ),
                ("standardize", MinMaxScaler(feature_range=(-1, 1))),


            ]),

            numeric_features
        )
    ]
)

In [32]:
#choice off models with initial hyperparameters, will be tuned later

models = {

    "LinearRegression":

        LinearRegression(),

    "DecisionTree":

        DecisionTreeRegressor(
            max_depth=8,
            random_state=42
        ),

    "XGBoost":

        XGBRegressor(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )
}

In [33]:
#metrics for evaluation
def rmse(y_true,y_pred):

    return np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

def mape(y_true,y_pred):

    y_true = np.array(y_true)

    y_pred = np.array(y_pred)

    mask = y_true != 0

    return (
        np.abs(
            (y_true[mask]-y_pred[mask])
            /
            y_true[mask]
        ).mean()
        *100
    )

In [62]:
# training function

def train_models(df,target):

    X = df[
        categorical_features +
        numeric_features
    ]

    y = df[target]

    tscv = TimeSeriesSplit( #creates dataset split object on which n-fold cross validation is performed
        n_splits=5
    )

    fitted = {}

    results = []

    for name, model in models.items():

        pipe = Pipeline([

            ("prep",preprocessor),

            ("model", clone(model))

        ])

        maes = []
        rmses = []
        mapes = []

        for train_idx,test_idx in tscv.split(X):

            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]

            y_train = y.iloc[train_idx]
            y_test = y.iloc[test_idx]

            pipe.fit(
                X_train,
                y_train
            )

            preds = pipe.predict(
                X_test
            )

            maes.append(
                mean_absolute_error(
                    y_test,
                    preds
                )
            )

            rmses.append(
                rmse(
                    y_test,
                    preds
                )
            )

            mapes.append(
                mape(
                    y_test,
                    preds
                )
            )

        results.append({

            "Target":target,

            "Model":name,

            "MAE":np.mean(maes),

            "RMSE":np.mean(rmses),

            "MAPE":np.mean(mapes)

        })

        pipe.fit(X,y)

        fitted[name] = pipe

    return fitted,pd.DataFrame(results)

In [63]:
# df
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()


In [64]:
#area and yield model training

area_df = df[
    df["Area harvested"].notna()
].copy()

# print(area_df)

area_models, area_scores = train_models(
    area_df,
    "Area harvested"
)

yield_df = df[
    df["Yield"].notna()
].copy()

# print(yield_df)

yield_models, yield_scores = train_models(
    yield_df,
    "Yield"
)

print("Yield target")
print(yield_df["Yield"].head())

print("Area target")
print(area_df["Area harvested"].head())

print("Yield mean:", yield_df["Yield"].mean())
print("Area mean:", area_df["Area harvested"].mean())

Yield target
20700    396.4
20783    410.0
20866    451.4
20949    470.4
21032    396.0
Name: Yield, dtype: float64
Area target
20700    500000.0
20783    500000.0
20866    500000.0
20949    500000.0
21032    500000.0
Name: Area harvested, dtype: float64
Yield mean: 630.3276814386642
Area mean: 136202.75765360738


In [66]:
print(id(yield_models["XGBoost"].named_steps["model"]))
print(id(area_models["XGBoost"].named_steps["model"]))

128910628547632
128910628541104


In [67]:
print(yield_models["XGBoost"].predict(sample))
print(area_models["XGBoost"].predict(sample))

[398.30496]
[498637.9]


In [68]:
print(yield_df.shape)
print(area_df.shape)

print(yield_df["Yield"].equals(area_df["Area harvested"]))

(4671, 16)
(4671, 16)
False


In [69]:
#compare scores

all_scores = pd.concat(
    [
        yield_scores,
        area_scores
    ]
)

print(
    all_scores
    .sort_values(
        ["Target","MAPE"]
    )
)

           Target             Model           MAE          RMSE          MAPE
1  Area harvested      DecisionTree  1.093829e+04  3.071393e+04  1.770163e+02
2  Area harvested           XGBoost  5.649783e+03  1.688356e+04  2.204596e+02
0  Area harvested  LinearRegression  8.618111e+16  8.699706e+16  1.496299e+17
2           Yield           XGBoost  2.422998e+01  7.190366e+01  5.882105e+00
1           Yield      DecisionTree  5.272405e+01  1.213925e+02  1.136805e+01
0           Yield  LinearRegression  4.849303e+13  4.894009e+13  1.699086e+13


FORECASTS

In [70]:
yield_model = yield_models["XGBoost"]
area_model = area_models["XGBoost"]

In [71]:
#forecast function
def forecast_country(
    hist,
    yield_model,
    area_model,
    years=[2025,2026]
):

    hist = hist.copy()

    forecasts = []

    for year in years:

        row = {}

        row["Area"] = hist.iloc[-1]["Area"]
        # row["Item"] = hist.iloc[-1]["Item"]

        row["Year"] = year

        for col in [
            # "Production",
            "Yield",
            "Area harvested"
        ]:

            row[f"{col}_lag1"] = hist[col].iloc[-1]

            row[f"{col}_lag2"] = (
                hist[col].iloc[-2]
            )

            row[f"{col}_lag3"] = (
                hist[col].iloc[-3]
            )

            row[f"{col}_roll3"] = (
                hist[col]
                .tail(3)
                .mean()
            )

            row[f"{col}_roll5"] = (
                hist[col]
                .tail(5)
                .mean()
            )

            row[f"{col}_growth"] = (
                hist[col]
                .pct_change()
                .iloc[-1]
            )

        X_pred = pd.DataFrame([row])
        print("blalableala")
        print(X_pred)
        print("blalableala")
        
        yield_pred = (
            yield_model
            .predict(X_pred)[0]
        )

        area_pred = (
            area_model
            .predict(X_pred)[0]
        )
        print("khmkhmkhm")
        # production_pred = (
        #     yield_pred *
        #     area_pred
        # )
        print(yield_pred)
        print(area_pred)
        row["Yield"] = yield_pred
        row["Area harvested"] = area_pred
        # row["Production"] = production_pred

        forecasts.append(row)

        hist = pd.concat(
            [
                hist,
                pd.DataFrame([row])
            ],
            ignore_index=True
        )

    return pd.DataFrame(
        forecasts
    )

In [72]:
#forecasts for all countries

predictions = []

for (country), group in df.groupby(
    ["Area"]
):

    fc = forecast_country(
        group.sort_values("Year"),
        yield_model,
        area_model
    )

    predictions.append(fc)

predictions = pd.concat(
    predictions,
    ignore_index=True
)

# predictions.to_csv(
#     "forecast_2025_2026.csv",
#     index=False
# )

blalableala
     Area  Year  Yield_lag1  Yield_lag2  Yield_lag3  Yield_roll3  Yield_roll5  \
0  Angola  2025       343.8       340.6       337.3   340.566667       341.26   

   Yield_growth  Area harvested_lag1  Area harvested_lag2  \
0      0.009395              22060.0              18290.0   

   Area harvested_lag3  Area harvested_roll3  Area harvested_roll5  \
0              15432.0               18594.0               23992.8   

   Area harvested_growth  
0               0.206124  
blalableala
khmkhmkhm
345.64465
24599.328
blalableala
     Area  Year  Yield_lag1  Yield_lag2  Yield_lag3  Yield_roll3  Yield_roll5  \
0  Angola  2026  345.644653       343.8       340.6   343.348218   342.588931   

   Yield_growth  Area harvested_lag1  Area harvested_lag2  \
0      0.005365         24599.328125              22060.0   

   Area harvested_lag3  Area harvested_roll3  Area harvested_roll5  \
0              18290.0          21649.776042          22039.065625   

   Area harvested_growth  

In [91]:
predictions["Production"] = (predictions["Yield"] * predictions["Area harvested"])/1000

In [92]:
predictions

,Area,Year,Yield_lag1,Yield_lag2,Yield_lag3,Yield_roll3,Yield_roll5,Yield_growth,Area harvested_lag1,Area harvested_lag2,Area harvested_lag3,Area harvested_roll3,Area harvested_roll5,Area harvested_growth,Yield,Area harvested,Production
0,Angola,2025,343.800000,340.6,337.3,340.566667,341.260000,0.009395,22060.000000,18290.0,15432.0,18594.000000,23992.800000,0.206124,345.644653,24599.328125,8502.625977
1,Angola,2026,345.644653,343.8,340.6,343.348218,342.588931,0.005365,24599.328125,22060.0,18290.0,21649.776042,22039.065625,0.115110,351.930511,26854.539062,9450.931641
2,Belize,2025,1180.600000,1201.8,1217.9,1200.100000,1205.220000,-0.017640,76.000000,75.0,74.0,75.000000,74.400000,0.013333,1161.804688,390.595154,453.795288
3,Belize,2026,1161.804688,1180.6,1201.8,1181.401562,1193.240937,-0.015920,390.595154,76.0,75.0,180.531718,137.919031,4.139410,1148.000366,7464.547363,8569.302734
4,Benin,2025,194.500000,194.5,194.5,194.500000,194.460000,0.000000,284.000000,283.0,284.0,283.666667,283.400000,0.003534,193.898422,350.700134,68.000206
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,Yemen,2026,767.905273,714.7,669.0,717.201758,783.561055,0.074444,35520.273438,37139.0,36829.0,36496.091146,36821.454688,-0.043586,828.882263,32620.580078,27038.619141
158,Zambia,2025,1082.400000,1082.4,1083.1,1082.633333,1082.400000,0.000000,8126.000000,8118.0,8163.0,8135.666667,8119.600000,0.000985,1086.224976,8218.951172,8927.629883
159,Zambia,2026,1086.224976,1082.4,1082.4,1083.674992,1083.184995,0.003534,8218.951172,8126.0,8118.0,8154.317057,8144.590234,0.011439,1091.400391,8490.425781,9266.454102
160,Zimbabwe,2025,817.100000,971.5,1000.0,929.533333,939.700000,-0.158929,700.000000,685.0,681.0,688.666667,663.000000,0.021898,708.493103,815.093689,577.488220


In [99]:
YP2025 = predictions[predictions["Year"] == 2025]
YP2025 = YP2025[["Area","Production", "Area harvested", "Yield"]]
YP2025 = YP2025.sort_values("Production", ascending=False)  
YP2025

,Area,Production,Area harvested,Yield
8,Brazil,3.344489e+06,1.943920e+06,1720.487183
154,Viet Nam,2.115519e+06,7.082721e+05,2986.873535
24,Colombia,1.019703e+06,8.359802e+05,1219.769531
72,Indonesia,8.477300e+05,1.270518e+06,667.232056
50,Ethiopia,5.771736e+05,8.494367e+05,679.478027
...,...,...,...,...
124,Sao Tome and Principe,1.243338e+01,2.342208e+02,53.083992
132,Suriname,8.276700e+00,2.890867e+02,28.630516
98,New Caledonia,-3.470393e+01,-2.525638e+02,137.406586
12,Cabo Verde,-3.837234e+01,-1.608719e+02,238.527313


In [105]:
data_cof[data_cof["Year"] == 2024].sort_values("Production", ascending=False)
Y2024 = data_cof[data_cof["Year"] == 2024].sort_values("Production", ascending=False)
Y2024 = Y2024[["Area","Production", "Area harvested", "Yield"]]
# Y2024[Y2024["Area"] == "New Caledonia"]
Y2024

,Area,Production,Area harvested,Yield
128642,Brazil,3387724.00,1943977.0,1742.7
1103275,Viet Nam,2015289.00,678569.0,2969.9
231488,Colombia,839846.69,837685.0,1002.6
482877,Indonesia,807578.00,1269354.0,636.2
351120,Ethiopia,575696.18,862943.0,667.1
...,...,...,...,...
881341,Samoa,11.79,39.0,300.2
884563,Sao Tome and Principe,7.27,131.0,55.7
958532,Suriname,6.22,292.0,21.3
722701,New Caledonia,1.01,7.0,138.2


In [76]:
df[df["Year"] == 2024]

,Area,Year,Area harvested,Yield,Yield_lag1,Yield_lag2,Yield_lag3,Area harvested_lag1,Area harvested_lag2,Area harvested_lag3,Yield_roll3,Yield_roll5,Area harvested_roll3,Area harvested_roll5,Yield_growth,Area harvested_growth
25797,Angola,2024,22060.0,343.8,340.6,337.3,345.6,18290.0,15432.0,29814.0,341.166667,338.74,2.117867e+04,27586.6,0.009395,0.206124
92806,Belize,2024,76.0,1180.6,1201.8,1217.9,1204.1,75.0,74.0,74.0,1207.933333,1216.46,7.433333e+01,73.6,-0.017640,0.013333
98463,Benin,2024,284.0,194.5,194.5,194.5,194.4,283.0,284.0,283.0,194.466667,194.48,2.833333e+02,284.0,0.000000,0.003534
111599,Bolivia (Plurinational State of),2024,25119.0,918.2,924.8,895.4,917.9,25496.0,25954.0,25548.0,912.700000,925.64,2.566600e+04,25609.8,-0.007137,-0.014787
128642,Brazil,2024,1943977.0,1742.7,1757.5,1702.6,1629.2,1905239.0,1867392.0,1832508.0,1696.433333,1738.26,1.868380e+06,1865770.2,-0.008421,0.020332
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1097039,Venezuela (Bolivarian Republic of),2024,166453.0,350.1,351.5,355.5,303.4,166792.0,160839.0,177079.0,336.800000,340.72,1.682367e+05,165133.2,-0.003983,-0.002032
1103275,Viet Nam,2024,678569.0,2969.9,2933.9,2979.0,2824.6,666954.0,655921.0,653192.0,2912.500000,2841.24,6.586890e+05,647546.0,0.012270,0.017415
1109972,Yemen,2024,37139.0,714.7,669.0,641.2,1125.0,36829.0,37346.0,37273.0,811.733333,725.26,3.714933e+04,36479.8,0.068311,0.008417
1115123,Zambia,2024,8126.0,1082.4,1082.4,1083.1,1081.8,8118.0,8163.0,8097.0,1082.433333,1082.94,8.126000e+03,8153.8,0.000000,0.000985


In [104]:
df[df["Area"] == "New Caledonia"]

,Area,Year,Area harvested,Yield,Yield_lag1,Yield_lag2,Yield_lag3,Area harvested_lag1,Area harvested_lag2,Area harvested_lag3,Yield_roll3,Yield_roll5,Area harvested_roll3,Area harvested_roll5,Yield_growth,Area harvested_growth
719523,New Caledonia,1964,6000.0,406.2,375.0,354.2,375.0,6000.0,6000.0,6000.0,368.066667,368.066667,6000.000000,6000.0,0.083200,0.000000
719572,New Caledonia,1965,6000.0,366.7,406.2,375.0,354.2,6000.0,6000.0,6000.0,378.466667,377.600000,6000.000000,6000.0,-0.097243,0.000000
719621,New Caledonia,1966,5000.0,354.0,366.7,406.2,375.0,6000.0,6000.0,6000.0,382.633333,375.420000,6000.000000,6000.0,-0.034633,-0.166667
719670,New Caledonia,1967,4000.0,384.3,354.0,366.7,406.2,5000.0,6000.0,6000.0,375.633333,371.220000,5666.666667,5800.0,0.085593,-0.200000
719719,New Caledonia,1968,3000.0,290.0,384.3,354.0,366.7,4000.0,5000.0,6000.0,368.333333,377.240000,5000.000000,5400.0,-0.245381,-0.250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
722477,New Caledonia,2020,14.0,140.7,140.5,140.2,139.8,14.0,19.0,37.0,140.166667,139.220000,23.333333,28.4,0.001423,0.000000
722534,New Caledonia,2021,14.0,139.9,140.7,140.5,140.2,14.0,14.0,19.0,140.466667,140.200000,15.666667,21.8,-0.005686,0.000000
722591,New Caledonia,2022,11.0,138.1,139.9,140.7,140.5,14.0,14.0,14.0,140.366667,140.220000,14.000000,19.6,-0.012866,-0.214286
722648,New Caledonia,2023,9.0,138.2,138.1,139.9,140.7,11.0,14.0,14.0,139.566667,139.880000,13.000000,14.4,0.000724,-0.181818


In [217]:
df_2024 = Y2024.rename(columns={"Production": "2024"})
df_2025 = YP2025.rename(columns={"Production": "2025"})

df_2024 = df_2024.drop(columns=["Area harvested", "Yield"])
df_2025 = df_2025.drop(columns=["Area harvested", "Yield"])

merged = pd.merge(df_2024, df_2025, on="Area", suffixes=("_2024", "_2025"))

num_cols = merged.select_dtypes(include="number").columns
merged[num_cols] = merged[num_cols].clip(lower=0)

merged["Increase"] = (merged["2025"] - merged["2024"]) / merged["2024"] * 100
pd.set_option('display.float_format', '{:.2f}'.format)
merged = merged.round(2)
std_scaler = StandardScaler()
# merged[['Increase']] = std_scaler.fit_transform(merged[['Increase']])
merged

upper = merged["Increase"].quantile(0.99)

merged["Increase_clipped"] = merged["Increase"].clip(upper=upper)

In [218]:
merged["Increase_clipped"]

0       -1.28
1        4.97
2       21.42
3        4.97
4        0.26
       ...   
74     567.53
75      71.02
76      33.07
77    -100.00
78   11014.69
Name: Increase_clipped, Length: 79, dtype: float64

In [219]:
import plotly.express as px

Increase = merged[["Area","Increase_clipped"]]

fig = px.choropleth(
    Increase,
    locations="Area",
    locationmode="country names",
    color="Increase_clipped",
    color_continuous_scale="Viridis",
    range_color=(merged["Increase_clipped"].min(), merged["Increase_clipped"].max()),
    title="Global Distribution"
)

fig.show()

In [215]:
Increase2 = merged[["Area","Increase"]]

fig = px.choropleth(
    Increase2,
    locations="Area",
    locationmode="country names",
    color="Increase",
    color_continuous_scale="Viridis",
    range_color=(
        Increase2["Increase"].quantile(0.1),
        Increase2["Increase"].quantile(0.80)
    ),
    title="Global Distribution (10–90% range)"
)

fig.show()